# 501: Probability of No Further Warming - Scenario Group Analysis

This notebook calculates the probability that warming declines after net-zero emissions are achieved,
comparing three groups of scenarios:
1. **NZCO2**: Scenarios with "NZCO2" in the label
2. **NZKyoto**: Scenarios with "NZKyoto" in the label
3. **Base**: All other scenarios (no NZCO2 or NZKyoto in label)

**Criteria for decline:**
- Peak warming occurs before 2100
- Delta warming (peak - 2100) > 0°C

In [1]:
import os
from pathlib import Path

import dotenv
import pandas as pd

dotenv.load_dotenv()

True

In [2]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

# Guardrails for classifying decline
PEAK_YEAR_THRESHOLD = 2100  # Peak must occur before this year
DELTA_WARMING_THRESHOLD = 0.0  # Minimum decline in °C to count as decline

In [3]:
# Input metrics file - uses combined manifest from 401
MANIFEST_ID = "2026-05-27_09-04"
METRICS_FILE = OUTPUT_FOLDER / f"500_gsat_metrics_{MANIFEST_ID}.csv"

## Load and Filter Data

In [4]:
# Load metrics
metrics_df = pd.read_csv(METRICS_FILE)
print(f"Total rows: {len(metrics_df)}")
metrics_df.head()

Total rows: 3051600


,model,scenario,magicc_flag,run_id,gsat_peak,gsat_peak_year,gsat_2100,delta_warming
0,AIM/CGE 2.0,SSP1-19,ALL,0,1.613,2049,1.411,0.202
1,AIM/CGE 2.0,SSP1-19,ALL,1,2.593,2093,2.577,0.015
2,AIM/CGE 2.0,SSP1-19,ALL,2,1.798,2049,1.577,0.221
3,AIM/CGE 2.0,SSP1-19,ALL,3,1.380,2038,1.074,0.306
4,AIM/CGE 2.0,SSP1-19,ALL,4,1.645,2048,1.396,0.249


In [5]:
# Filter for ALL magicc_flag only
all_df = metrics_df[metrics_df["magicc_flag"] == "ALL"].copy()
print(f"Filtered to magicc_flag=ALL: {len(all_df)} rows")

# Classify scenarios into three groups
all_df["scenario_group"] = all_df["scenario"].apply(
    lambda x: "NZCO2" if "NZCO2" in x 
    else ("NZKyoto" if "NZKyoto" in x else "Base")
)

# Show group counts
group_counts = all_df.groupby("scenario_group").agg(
    n_rows=("run_id", "count"),
    n_scenarios=("scenario", "nunique")
)
print("\nScenario group summary:")
group_counts

Filtered to magicc_flag=ALL: 1017600 rows

Scenario group summary:


,n_rows,n_scenarios
scenario_group,,
Base,410400,364
NZCO2,410400,364
NZKyoto,196800,195


## Classify Ensemble Members

In [6]:
# Apply guardrails to classify each ensemble member as "decline" or not
all_df["is_decline"] = (
    (all_df["gsat_peak_year"] < PEAK_YEAR_THRESHOLD) &
    (all_df["delta_warming"] > DELTA_WARMING_THRESHOLD)
)

# Summary by group
decline_summary = all_df.groupby("scenario_group").agg(
    total_members=("is_decline", "count"),
    decline_members=("is_decline", "sum"),
)
decline_summary["no_decline_members"] = decline_summary["total_members"] - decline_summary["decline_members"]
decline_summary["pct_decline"] = (decline_summary["decline_members"] / decline_summary["total_members"] * 100).round(1)
print("Decline classification by scenario group:")
decline_summary

Decline classification by scenario group:


,total_members,decline_members,no_decline_members,pct_decline
scenario_group,,,,
Base,410400,382755,27645,93.3
NZCO2,410400,348149,62251,84.8
NZKyoto,196800,190683,6117,96.9


## Calculate Probability Per Scenario

In [7]:
# Group by model, scenario, and scenario_group, calculate probability
decline_prob = all_df.groupby(["scenario_group", "model", "scenario"]).agg(
    n_ensemble=pd.NamedAgg(column="run_id", aggfunc="count"),
    n_decline=pd.NamedAgg(column="is_decline", aggfunc="sum"),
).reset_index()

# Calculate probability
decline_prob["prob_decline"] = decline_prob["n_decline"] / decline_prob["n_ensemble"]
decline_prob["prob_decline_pct"] = (decline_prob["prob_decline"] * 100).round(1)

print(f"Calculated probabilities for {len(decline_prob)} scenarios")
decline_prob.head(10)

Calculated probabilities for 1696 scenarios


,scenario_group,model,scenario,n_ensemble,n_decline,prob_decline,prob_decline_pct
0,Base,AIM/CGE 2.0,SSP1-19,600,596,0.993333,99.3
1,Base,AIM/CGE 2.0,SSP2-19,600,600,1.000000,100.0
2,Base,AIM/CGE 2.0,SSP2-26,600,579,0.965000,96.5
3,Base,AIM/CGE 2.0,SSP3-34,600,425,0.708333,70.8
4,Base,AIM/CGE 2.0,SSP4-26,600,589,0.981667,98.2
5,Base,AIM/CGE 2.0,SSP5-26,600,566,0.943333,94.3
6,Base,AIM/CGE 2.1,CD-LINKS-NDC2030i_1000,600,285,0.475000,47.5
7,Base,AIM/CGE 2.1,CD-LINKS-NPi2020_400,600,587,0.978333,97.8
8,Base,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Cost-Effective,600,370,0.616667,61.7
9,Base,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Enablers & Institutional,600,552,0.920000,92.0


In [8]:
# Create structured summary table for each group
def create_group_summary(df, group_name):
    """Create summary statistics for a scenario group."""
    group_data = df[df["scenario_group"] == group_name]
    
    summary = {
        "Group": group_name,
        "N Scenarios": len(group_data),
        "Median Prob (%)": group_data["prob_decline_pct"].median().round(1),
        "34th Pctl (%)": group_data["prob_decline_pct"].quantile(0.34).round(1),
        "67th Pctl (%)": group_data["prob_decline_pct"].quantile(0.67).round(1),
        "5th Pctl (%)": group_data["prob_decline_pct"].quantile(0.05).round(1),
        "95th Pctl (%)": group_data["prob_decline_pct"].quantile(0.95).round(1),
    }
    return summary

# Create summary for each group
summaries = []
for group in ["Base", "NZCO2", "NZKyoto"]:
    summaries.append(create_group_summary(decline_prob, group))

summary_table = pd.DataFrame(summaries)
print("Summary Table: Probability of Decline by Scenario Group")
summary_table

Summary Table: Probability of Decline by Scenario Group


,Group,N Scenarios,Median Prob (%),34th Pctl (%),67th Pctl (%),5th Pctl (%),95th Pctl (%)
0,Base,684,97.3,93.7,99.3,75.5,100.0
1,NZCO2,684,87.8,84.0,92.0,58.5,98.7
2,NZKyoto,328,97.5,96.7,98.7,90.7,100.0


## Detailed Results by Group

In [9]:
# Base scenarios (no NZCO2 or NZKyoto)
print("BASE SCENARIOS (no net-zero suffix)")
print("=" * 50)
base_results = decline_prob[decline_prob["scenario_group"] == "Base"][
    ["model", "scenario", "n_ensemble", "n_decline", "prob_decline_pct"]
].sort_values("prob_decline_pct", ascending=False)
base_results

BASE SCENARIOS (no net-zero suffix)


,model,scenario,n_ensemble,n_decline,prob_decline_pct
683,WITCH-GLOBIOM 4.4,CD-LINKS-NPi2020_400,600,600,100.0
483,REMIND 2.1,LeastTotalCost-LTC-brkLR15-SSP5-P50,600,600,100.0
520,REMIND 3.0,NAVIGATE Demand-1.5°C-tec_u,600,600,100.0
519,REMIND 3.0,NAVIGATE Demand-1.5°C-ref,600,600,100.0
518,REMIND 3.0,NAVIGATE Demand-1.5°C-ele_u,600,600,100.0
...,...,...,...,...,...
190,IMAGE 3.2,ENGAGE-Feasibility-Maximum-Effort/Institutional,600,307,51.2
170,IMAGE 3.0.1,EMF30-D-CO2-Only,600,296,49.3
6,AIM/CGE 2.1,CD-LINKS-NDC2030i_1000,600,285,47.5
146,IMAGE 3.0,ENGAGE-INDCi2030-1000,600,283,47.2


In [10]:
# NZKyoto scenarios
print("NZKyoto SCENARIOS")
print("=" * 50)
nzkyoto_results = decline_prob[decline_prob["scenario_group"] == "NZKyoto"][
    ["model", "scenario", "n_ensemble", "n_decline", "prob_decline_pct"]
].sort_values("prob_decline_pct", ascending=False)
nzkyoto_results

NZKyoto SCENARIOS


,model,scenario,n_ensemble,n_decline,prob_decline_pct
1598,REMIND 1.7,ADVANCE-2030-1.5°C-2100_NZKyoto,600,600,100.0
1607,REMIND 2.1,LeastTotalCost-LTC-brkLR15-SSP1-P50_NZKyoto,600,600,100.0
1453,IMAGE 3.2,SSP2021-SSP2-SPA2-19-Renewables_NZKyoto,600,600,100.0
1599,REMIND 1.7,ADVANCE-2030-Price 1.5°C_NZKyoto,600,600,100.0
1600,REMIND 1.7,ADVANCE-2030-well-below-2°C_NZKyoto,600,600,100.0
...,...,...,...,...,...
1672,WITCH 5.0,ENGAGE-INDCi2030-1200f-NDCp_NZKyoto,600,532,88.7
1477,MESSAGE-GLOBIOM 1.0,SSP1-34_NZKyoto,600,521,86.8
1673,WITCH 5.0,ENGAGE-INDCi2030-1200f_NZKyoto,600,516,86.0
1447,IMAGE 3.2,SSP2021-SSP1-SPA1-26-Default_NZKyoto,600,504,84.0


In [11]:
# NZCO2 scenarios
print("NZCO2 SCENARIOS")
print("=" * 50)
nzco2_results = decline_prob[decline_prob["scenario_group"] == "NZCO2"][
    ["model", "scenario", "n_ensemble", "n_decline", "prob_decline_pct"]
].sort_values("prob_decline_pct", ascending=False)
nzco2_results

NZCO2 SCENARIOS


,model,scenario,n_ensemble,n_decline,prob_decline_pct
1295,REMIND-MAgPIE 3.2-4.6,SHAPE-SSP1-1.5°C_NZCO2,600,600,100.0
1252,REMIND-MAgPIE 2.1-4.2,CEMICS-SSP1-1.5°C-fullCDR_NZCO2,600,600,100.0
1289,REMIND-MAgPIE 3.2-4.6,SHAPE-SDP-EI-1.5°C [with Climate Change Impact...,600,600,100.0
1290,REMIND-MAgPIE 3.2-4.6,SHAPE-SDP-EI-1.5°C_NZCO2,600,600,100.0
1291,REMIND-MAgPIE 3.2-4.6,SHAPE-SDP-MC-1.5°C [with Climate Change Impact...,600,600,100.0
...,...,...,...,...,...
1315,TIAM-ECN 1.1,ENGAGE-NPi2020-1200f_NZCO2,600,192,32.0
1313,TIAM-ECN 1.1,ENGAGE-NPi2020-1000f-COV_NZCO2,600,191,31.8
690,AIM/CGE 2.1,CD-LINKS-NDC2030i_1000_NZCO2,600,178,29.7
1314,TIAM-ECN 1.1,ENGAGE-NPi2020-1000f_NZCO2,600,177,29.5
